# Cattle Disease Classifier Training

Run all cells in order. The final cell creates `cattle_disease_model.keras` for `app.py`.

In [ ]:
import tensorflow as tf
import sklearn

print("TensorFlow:", tf.__version__)
print("scikit-learn installed")

TensorFlow: 2.21.0
scikit-learn installed


In [3]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install tensorflow scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path
import json
import random
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

PROJECT_DIR = Path(r'C:\Users\petta\Desktop\cattle dataset\Cows datasets')
CLASS_NAMES = ['foot-and-mouth', 'healthy', 'lumpy']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}')
print('Project folder:', PROJECT_DIR)
print('TensorFlow version:', tf.__version__)

Project folder: C:\Users\petta\Desktop\cattle dataset\Cows datasets
TensorFlow version: 2.21.0


In [6]:
# Read each class and make a reproducible 70% / 15% / 15% split.
files_by_class = {}
for class_name in CLASS_NAMES:
    class_folder = PROJECT_DIR / class_name
    if not class_folder.is_dir():
        raise FileNotFoundError(f'Missing class folder: {class_folder}')
    files = sorted(path for path in class_folder.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    if len(files) < 10:
        raise ValueError(f'{class_name} needs at least 10 images; found {len(files)}')
    files_by_class[class_name] = files
    print(f'{class_name}: {len(files)} images')

train_paths, train_labels = [], []
validation_paths, validation_labels = [], []
test_paths, test_labels = [], []
rng = random.Random(SEED)
for label, class_name in enumerate(CLASS_NAMES):
    paths = files_by_class[class_name].copy()
    rng.shuffle(paths)
    train_end = int(0.70 * len(paths))
    validation_end = train_end + int(0.15 * len(paths))
    for path in paths[:train_end]:
        train_paths.append(str(path)); train_labels.append(label)
    for path in paths[train_end:validation_end]:
        validation_paths.append(str(path)); validation_labels.append(label)
    for path in paths[validation_end:]:
        test_paths.append(str(path)); test_labels.append(label)
print(f'Train: {len(train_paths)}, validation: {len(validation_paths)}, test: {len(test_paths)}')

foot-and-mouth: 746 images
healthy: 1291 images
lumpy: 1206 images
Train: 2269, validation: 484, test: 490


In [7]:
# Dataset preparation. The preprocessing is exactly the same as app.py.
AUTOTUNE = tf.data.AUTOTUNE
def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return dataset.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels, training=True)
validation_ds = make_dataset(validation_paths, validation_labels)
test_ds = make_dataset(test_paths, test_labels)
class_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))
class_weights = {index: len(train_labels) / (len(CLASS_NAMES) * count) for index, count in enumerate(class_counts)}
print('Class weights:', class_weights)

Class weights: {0: np.float64(1.4489144316730524), 1: np.float64(0.837578442229605), 2: np.float64(0.8961295418641391)}


In [8]:
# Transfer-learning model. Internet is required once to download ImageNet weights.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
])
base_model = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False
inputs = tf.keras.Input(shape=(224, 224, 3))
x = augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
# First training stage.
CHECKPOINT_PATH = PROJECT_DIR / 'best_cattle_disease_model.keras'
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.3, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor='val_loss', save_best_only=True),
]
history = model.fit(train_ds, validation_data=validation_ds, epochs=15, class_weight=class_weights, callbacks=callbacks)

Epoch 1/15


71/71 ━━━━━━━━━━━━━━━━━━━━ 62s 782ms/step - accuracy: 0.3636 - loss: 1.3710 - val_accuracy: 0.5227 - val_loss: 1.0040 - learning_rate: 1.0000e-04
Epoch 2/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 55s 770ms/step - accuracy: 0.4896 - loss: 1.0742 - val_accuracy: 0.6860 - val_loss: 0.7736 - learning_rate: 1.0000e-04
Epoch 3/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 55s 768ms/step - accuracy: 0.6038 - loss: 0.8893 - val_accuracy: 0.7438 - val_loss: 0.6504 - learning_rate: 1.0000e-04
Epoch 4/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 58s 815ms/step - accuracy: 0.6593 - loss: 0.7877 - val_accuracy: 0.7810 - val_loss: 0.5776 - learning_rate: 1.0000e-04
Epoch 5/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 58s 814ms/step - accuracy: 0.6955 - loss: 0.7071 - val_accuracy: 0.8017 - val_loss: 0.5314 - learning_rate: 1.0000e-04
Epoch 6/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 58s 814ms/step - accuracy: 0.7201 - loss: 0.6578 - val_accuracy: 0.8306 - val_loss: 0.4892 - learning_rate: 1.0000e-04
Epoch 7/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 84s 844ms/step - accuracy: 0.73

In [10]:

# Fine-tune the final 30 MobileNetV2 layers.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
fine_tune_history = model.fit(train_ds, validation_data=validation_ds, epochs=10, class_weight=class_weights, callbacks=callbacks)

Epoch 1/10


71/71 ━━━━━━━━━━━━━━━━━━━━ 81s 963ms/step - accuracy: 0.7739 - loss: 0.5403 - val_accuracy: 0.8822 - val_loss: 0.3072 - learning_rate: 1.0000e-05
Epoch 2/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 70s 985ms/step - accuracy: 0.8290 - loss: 0.4301 - val_accuracy: 0.8822 - val_loss: 0.2889 - learning_rate: 1.0000e-05
Epoch 3/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 81s 965ms/step - accuracy: 0.8550 - loss: 0.3768 - val_accuracy: 0.8905 - val_loss: 0.2761 - learning_rate: 1.0000e-05
Epoch 4/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 68s 955ms/step - accuracy: 0.8731 - loss: 0.3416 - val_accuracy: 0.8967 - val_loss: 0.2675 - learning_rate: 1.0000e-05
Epoch 5/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.8836 - loss: 0.3077 - val_accuracy: 0.8988 - val_loss: 0.2587 - learning_rate: 1.0000e-05
Epoch 6/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.8960 - loss: 0.2849 - val_accuracy: 0.9050 - val_loss: 0.2513 - learning_rate: 1.0000e-05
Epoch 7/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 73s 1s/step - accuracy: 0.8872 - loss

In [11]:

# Evaluate and export the exact files used by app.py.
best_model = tf.keras.models.load_model(CHECKPOINT_PATH)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred = np.argmax(best_model.predict(test_ds, verbose=0), axis=1)
print(f'Test accuracy: {test_accuracy:.2%}')
print(f'Test loss: {test_loss:.4f}')
print('Confusion matrix:\n', confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

MODEL_PATH = PROJECT_DIR / 'cattle_disease_model.keras'
LABELS_PATH = PROJECT_DIR / 'class_names.json'
best_model.save(MODEL_PATH)
LABELS_PATH.write_text(json.dumps(CLASS_NAMES, indent=2), encoding='utf-8')
print(f'Saved model: {MODEL_PATH}')
print(f'Saved labels: {LABELS_PATH}')

Test accuracy: 88.57%
Test loss: 0.3000
Confusion matrix:
 [[111   2   0]
 [ 19 163  13]
 [  6  16 160]]
                precision    recall  f1-score   support

foot-and-mouth       0.82      0.98      0.89       113
       healthy       0.90      0.84      0.87       195
         lumpy       0.92      0.88      0.90       182

      accuracy                           0.89       490
     macro avg       0.88      0.90      0.89       490
  weighted avg       0.89      0.89      0.89       490

Saved model: C:\Users\petta\Desktop\cattle dataset\Cows datasets\cattle_disease_model.keras
Saved labels: C:\Users\petta\Desktop\cattle dataset\Cows datasets\class_names.json
